# Experiment 1: Automated Machine Learning Workflow
This notebook implements a complete end-to-end automated machine learning data pipeline.

### Associated Machine Learning Tasks:
1. **Iris Dataset:** Supervised Learning -> Multi-class Classification (predicting Iris flower species).
2. **Diabetes Dataset:** Supervised Learning -> Binary Classification (predicting diabetes onset).
3. **Loan Dataset:** Supervised Learning -> Binary Classification (predicting loan approval status).
4. **Email Spam Dataset:** Supervised Learning -> Binary Classification / NLP (identifying spam vs. ham).
5. **MNIST Dataset:** Supervised Learning -> Multi-class Classification / Image Recognition (recognizing handwritten digits 0-9).

In [ ]:
import pandas as pd
import numpy as np
import os
import struct
from sklearn.preprocessing import StandardScaler

def smart_loader(data_source):
    # Identifies the file type and loads it appropriately
    
    # 1. Handle Partitioned Datasets (Dictionaries)
    if isinstance(data_source, dict):
        print("Partitioned dataset detected. Loading splits...")
        loaded_splits = {}
        for split_name, file_path in data_source.items():
            # Recursively call the loader for each file
            data, data_type = smart_loader(file_path)
            loaded_splits[split_name] = data
        return loaded_splits, f"{data_type}_split"

    # 2. File Existence Check (Fails fast if the path is wrong)
    if not os.path.exists(data_source):
        raise FileNotFoundError(f"Local file not found: {data_source}")

    # Extract extension/filename for single files
    _, ext = os.path.splitext(data_source)
    ext = ext.lower()
    basename = os.path.basename(data_source).lower()

    # 3. Handle Known Tabular Formats
    if ext in ['.csv', '.data']:
        print(f"Loading CSV/data file: {os.path.basename(data_source)}")
        if 'iris' in basename:
            # Iris dataset doesn't have a header in the raw file
            return pd.read_csv(data_source, header=None, names=['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class']), "tabular"
        return pd.read_csv(data_source), "tabular"
        
    elif ext in ['.xls', '.xlsx']:
        print(f"Loading Excel file: {os.path.basename(data_source)}")
        return pd.read_excel(data_source), "tabular"

    # 4. Handle Raw Binary MNIST Files (.ubyte)
    elif 'ubyte' in ext or 'ubyte' in data_source:
        print(f"Detected raw byte file: {os.path.basename(data_source)}")
        with open(data_source, 'rb') as f:
            magic_number = struct.unpack('>I', f.read(4))[0]
            if magic_number == 2051: # Images
                num_items, rows, cols = struct.unpack('>III', f.read(12))
                data = np.fromfile(f, dtype=np.uint8).reshape(num_items, rows, cols)
                return data, "mnist_images"
            elif magic_number == 2049: # Labels
                num_items = struct.unpack('>I', f.read(4))[0]
                data = np.fromfile(f, dtype=np.uint8)
                return data, "mnist_labels"
            else:
                raise ValueError(f"Unknown MNIST magic number: {magic_number}")

    # 5. Total Failure
    else:
        raise ValueError(f"Completely unsupported local format: {ext}")

In [ ]:
# --- Test Ingestion of All Datasets ---
print("=== Loading Iris ===")
iris_data, iris_flag = smart_loader('Datasets/iris/bezdekIris.data')

print("\n=== Loading Diabetes ===")
diabetes_data, diabetes_flag = smart_loader('Datasets/Diabetes_Dataset/diabetes.csv')

print("\n=== Loading Loan Dataset (Dictionary Split) ===")
loan_splits = {
    'train': 'Datasets/Loan_Amount_Dataset/loan-train.csv',
    'test': 'Datasets/Loan_Amount_Dataset/loan-test.csv'
}
loan_data, loan_flag = smart_loader(loan_splits)

print("\n=== Loading Email Spam ===")
email_data, email_flag = smart_loader('Datasets/Email_Spam_Dataset/emails.csv')

print("\n=== Loading MNIST (Dictionary Split) ===")
mnist_splits = {
    'train_images': 'Datasets/MNIST_Dataset/train-images.idx3-ubyte',
    'train_labels': 'Datasets/MNIST_Dataset/train-labels.idx1-ubyte',
    'test_images': 'Datasets/MNIST_Dataset/t10k-images.idx3-ubyte',
    'test_labels': 'Datasets/MNIST_Dataset/t10k-labels.idx1-ubyte'
}
mnist_data, mnist_flag = smart_loader(mnist_splits)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec

def automated_eda(data, data_type):
    """
    Generates summary statistics and visualizations based on data type.
    """
    if "split" in data_type:
        print(f"Partitioned dataset detected with flag: {data_type}")
        has_images = any("image" in k for k in data.keys())
        has_labels = any("label" in k for k in data.keys())
        if "mnist" in data_type and has_images and has_labels:
            img_key = next((k for k in data.keys() if "images" in k or "image" in k), None)
            lbl_key = next((k for k in data.keys() if "labels" in k or "label" in k), None)
            if img_key and lbl_key:
                images, labels = data[img_key], data[lbl_key]
                print(f"\n=== Combined MNIST Visualization ({img_key} & {lbl_key}) ===")
                print(f"Images Shape: {images.shape}, Labels Shape: {labels.shape}")
                print(f"Pixel Range: [{images.min()}, {images.max()}]")
                
                fig = plt.figure(figsize=(15, 6))
                gs = gridspec.GridSpec(1, 2, width_ratios=[1.2, 1.0])
                
                # Left: Label Distribution
                ax_lbl = fig.add_subplot(gs[0])
                unique, counts = np.unique(labels, return_counts=True)
                sns.barplot(x=unique, y=counts, palette='viridis', ax=ax_lbl)
                ax_lbl.set_title("MNIST Class Distribution")
                ax_lbl.set_xlabel("Digit Class")
                ax_lbl.set_ylabel("Frequency")
                
                # Right: Grid of 3x3 digits
                gs_right = gridspec.GridSpecFromSubplotSpec(3, 3, subplot_spec=gs[1])
                for i in range(9):
                    ax_img = fig.add_subplot(gs_right[i])
                    idx = np.random.randint(0, len(images))
                    ax_img.imshow(images[idx], cmap='gray')
                    ax_img.axis('off')
                    ax_img.set_title(f"Idx: {idx}", fontsize=8)
                
                plt.suptitle("MNIST Dataset Summary Dashboard", fontsize=16)
                plt.tight_layout()
                plt.show()
                return
        
        for name, split in data.items():
            print(f"\n--- Exploring Split: {name} ---")
            if isinstance(split, pd.DataFrame):
                automated_eda(split, "tabular")
            elif isinstance(split, np.ndarray):
                m_type = "mnist_images" if len(split.shape) == 3 else "mnist_labels"
                automated_eda(split, m_type)

    elif "tabular" in data_type:
        print("=== TABULAR SUMMARY STATISTICS ===")
        print(data.info())
        print("\n--- Descriptive Statistics ---")
        print(data.describe())
        
        # Isolate numerical columns
        num_cols = data.select_dtypes(include=[np.number]).columns
        
        print("\n=== TABULAR VISUALIZATIONS ===")
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # Subplot 1: Missing Data Profile
        sns.heatmap(data.isnull(), yticklabels=False, cbar=False, cmap='viridis', ax=axes[0])
        axes[0].set_title("Missing Data Profile (Yellow = Missing)")
        
        # Subplot 2: Correlation Heatmap
        if len(num_cols) > 0:
            sns.heatmap(data[num_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f", ax=axes[1])
            axes[1].set_title("Feature Correlation Matrix")
        else:
            axes[1].text(0.5, 0.5, 'No Numerical Columns', ha='center', va='center')
            axes[1].set_title("Feature Correlation Matrix")
            
        # Subplot 3: Boxplot of Scaled Numerical Features
        if len(num_cols) > 0:
            # Scale features temporarily for clean boxplot representation
            temp_scaler = StandardScaler()
            temp_scaled = temp_scaler.fit_transform(data[num_cols].dropna())
            temp_df = pd.DataFrame(temp_scaled, columns=num_cols)
            sns.boxplot(data=temp_df, ax=axes[2], palette='Set2')
            axes[2].set_title("Boxplot of Scaled Numerical Features")
            axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45, ha='right')
        else:
            axes[2].text(0.5, 0.5, 'No Numerical Columns', ha='center', va='center')
            axes[2].set_title("Boxplot of Features")
            
        plt.tight_layout()
        plt.show()
        
        # Scatter Plot (First two numerical features if they exist)
        if len(num_cols) >= 2:
            plt.figure(figsize=(8, 4.5))
            sns.scatterplot(data=data, x=num_cols[0], y=num_cols[1], hue=data.columns[-1], palette='viridis')
            plt.title(f"Scatter Plot: {num_cols[0]} vs {num_cols[1]}")
            plt.tight_layout()
            plt.show()
        
        # Histograms
        if len(num_cols) > 0:
            data[num_cols].hist(figsize=(12, 8), bins=20, edgecolor='black', color='skyblue')
            plt.suptitle("Numerical Feature Distributions")
            plt.tight_layout()
            plt.show()

    elif "mnist" in data_type:
        print("=== MNIST DATA STATISTICS ===")
        print(f"Data type: {data_type}")
        print(f"Data shape: {data.shape}")
        print(f"Data values range: [{data.min()}, {data.max()}]")
        
        if data_type == "mnist_images":
            print("\n=== MNIST IMAGE VISUALIZATION ===")
            fig, axes = plt.subplots(3, 3, figsize=(6, 6))
            for i, ax in enumerate(axes.flat):
                idx = np.random.randint(0, len(data))
                ax.imshow(data[idx], cmap='gray')
                ax.axis('off')
                ax.set_title(f"Idx: {idx}")
            plt.suptitle("Sample Digits Grid")
            plt.tight_layout()
            plt.show()
            
        elif data_type == "mnist_labels":
            print("\n=== MNIST LABEL VISUALIZATION ===")
            unique, counts = np.unique(data, return_counts=True)
            plt.figure(figsize=(8, 4))
            sns.barplot(x=unique, y=counts, palette='viridis')
            plt.title("MNIST Class Distribution")
            plt.xlabel("Digit Class")
            plt.ylabel("Frequency")
            plt.tight_layout()
            plt.show()

In [ ]:
# --- Run EDA on Datasets ---
print("=== Running EDA on Iris Dataset ===")
automated_eda(iris_data, iris_flag)

print("\n=== Running EDA on Diabetes Dataset ===")
automated_eda(diabetes_data, diabetes_flag)

print("\n=== Running EDA on MNIST (Train Images) ===")
automated_eda(mnist_data['train_images'], 'mnist_images')

print("\n=== Running EDA on MNIST (Train Labels) ===")
automated_eda(mnist_data['train_labels'], 'mnist_labels')

print("\n=== Running Combined MNIST Dashboard ===")
automated_eda(mnist_data, mnist_flag)

In [ ]:
from sklearn.preprocessing import StandardScaler

def preprocess_dataset(data, data_type, dataset_name=None):
    """
    Performs data cleaning, imputation, categorical encoding, and scaling.
    """
    if "split" in data_type:
        processed_splits = {}
        for name, split in data.items():
            print(f"\n--- Preprocessing Split: {name} ---")
            if isinstance(split, pd.DataFrame):
                processed_splits[name] = preprocess_dataset(split, "tabular", dataset_name=dataset_name)
            elif isinstance(split, np.ndarray):
                m_type = "mnist_images" if len(split.shape) == 3 else "mnist_labels"
                processed_splits[name] = preprocess_dataset(split, m_type)
        return processed_splits

    elif "tabular" in data_type:
        df = data.copy()
        
        # 1. Handle dataset-specific medical placeholders
        if dataset_name and "diabetes" in dataset_name.lower():
            # In diabetes, zeros in certain columns are invalid
            invalid_zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
            for col in invalid_zero_cols:
                if col in df.columns:
                    df[col] = df[col].replace(0, np.nan)
            print("Diabetes: Replaced invalid zeros with NaN.")
            
        # 2. Impute missing values
        # Numeric imputation (median)
        num_cols = df.select_dtypes(include=[np.number]).columns
        for col in num_cols:
            if df[col].isnull().sum() > 0:
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val)
                print(f"Imputed missing numeric values in '{col}' with median: {median_val}")
                
        # Categorical imputation (mode)
        cat_cols = df.select_dtypes(exclude=[np.number]).columns
        for col in cat_cols:
            if df[col].isnull().sum() > 0:
                mode_val = df[col].mode()[0]
                df[col] = df[col].fillna(mode_val)
                print(f"Imputed missing categorical values in '{col}' with mode: {mode_val}")
                
        # 3. Categorical encoding
        # Refresh cat_cols after imputation
        cat_cols = df.select_dtypes(exclude=[np.number]).columns
        cols_to_encode = [c for c in cat_cols if not c.lower().endswith('_id')]
        for col in cols_to_encode:
            df[col] = df[col].astype('category').cat.codes
            print(f"Encoded categorical column '{col}' to category codes.")
            
        # 4. Feature Scaling (Standardization)
        target_col = df.columns[-1]
        scale_cols = [c for c in num_cols if c != target_col and not c.lower().endswith('_id')]
        
        if len(scale_cols) > 0:
            scaler = StandardScaler()
            df[scale_cols] = scaler.fit_transform(df[scale_cols])
            print(f"Standardized numerical columns: {scale_cols}")
            
        return df
        
    elif "mnist" in data_type:
        if data_type == "mnist_images":
            # 1. Normalize pixel values [0, 255] -> [0.0, 1.0]
            processed_data = data.astype(np.float32) / 255.0
            print("MNIST Images: Normalized pixel values to [0.0, 1.0]")
            
            # 2. Flatten 2D image arrays into 1D vectors (N x 784)
            num_images, rows, cols = processed_data.shape
            processed_data = processed_data.reshape(num_images, rows * cols)
            print(f"MNIST Images: Flattened 2D images {rows}x{cols} to 1D vectors of size {rows*cols}")
            return processed_data
            
        elif data_type == "mnist_labels":
            print("MNIST Labels: No preprocessing needed.")
            return data

In [ ]:
# --- Run Preprocessing on All Datasets ---
print("=== Preprocessing Iris Dataset ===")
processed_iris = preprocess_dataset(iris_data, iris_flag)
print("Processed Iris Shape:", processed_iris.shape)
print(processed_iris.head(2))

print("\n=== Preprocessing Diabetes Dataset ===")
processed_diabetes = preprocess_dataset(diabetes_data, diabetes_flag, dataset_name="Diabetes")
print("Processed Diabetes Shape:", processed_diabetes.shape)
print(processed_diabetes.head(2))

print("\n=== Preprocessing Loan Splits ===")
processed_loan = preprocess_dataset(loan_data, loan_flag)
print("Processed Loan Train Shape:", processed_loan['train'].shape)
print("Processed Loan Test Shape:", processed_loan['test'].shape)
print(processed_loan['train'].head(2))

print("\n=== Preprocessing Email Spam ===")
processed_email = preprocess_dataset(email_data, email_flag)
print("Processed Email Shape:", processed_email.shape)

print("\n=== Preprocessing MNIST Splits ===")
processed_mnist = preprocess_dataset(mnist_data, mnist_flag)
print("Processed MNIST Train Images Shape:", processed_mnist['train_images'].shape)
print("Processed MNIST Train Images Min/Max:", processed_mnist['train_images'].min(), "/", processed_mnist['train_images'].max())

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split

def select_features(X, y, k=4):
    """
    Selects top k features using SelectKBest with ANOVA F-value (f_classif).
    """
    selector = SelectKBest(score_func=f_classif, k=min(k, X.shape[1]))
    X_new = selector.fit_transform(X, y)
    selected_indices = selector.get_support(indices=True)
    selected_features = X.columns[selected_indices] if isinstance(X, pd.DataFrame) else list(range(X.shape[1]))
    scores = selector.scores_[selected_indices]
    
    print("--- Feature Selection Scores ---")
    for f, s in zip(selected_features, scores):
        print(f"Feature: {f:<25} | F-Score: {s:.2f}")
        
    return X_new, list(selected_features)

def split_dataset(X, y, val_size=0.15, test_size=0.15):
    """
    Splits dataset into Train, Validation, and Test sets.
    """
    rem_size = val_size + test_size
    X_train, X_rem, y_train, y_rem = train_test_split(X, y, test_size=rem_size, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_rem, y_rem, test_size=test_size/rem_size, random_state=42, stratify=y_rem)
    
    print(f"Split completed - Train: {X_train.shape[0]} samples, Val: {X_val.shape[0]} samples, Test: {X_test.shape[0]} samples")
    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
# --- Run Feature Selection and Data Splitting ---

# 1. Iris dataset
print("=== Processing Iris Dataset ===")
X_iris = processed_iris.iloc[:, :-1]
y_iris = processed_iris.iloc[:, -1]
X_iris_sel, iris_features = select_features(X_iris, y_iris, k=3)
X_iris_tr, X_iris_val, X_iris_te, y_iris_tr, y_iris_val, y_iris_te = split_dataset(X_iris_sel, y_iris)

# 2. Diabetes dataset
print("\n=== Processing Diabetes Dataset ===")
X_diab = processed_diabetes.iloc[:, :-1]
y_diab = processed_diabetes.iloc[:, -1]
X_diab_sel, diab_features = select_features(X_diab, y_diab, k=5)
X_diab_tr, X_diab_val, X_diab_te, y_diab_tr, y_diab_val, y_diab_te = split_dataset(X_diab_sel, y_diab)

# 3. Loan dataset (using train split for manual partition)
print("\n=== Processing Loan Dataset ===")
X_loan_train = processed_loan['train'].drop(columns=['Loan_ID', 'Loan_Status'])
y_loan_train = processed_loan['train']['Loan_Status']
X_loan_tr, X_loan_val, X_loan_te, y_loan_tr, y_loan_val, y_loan_te = split_dataset(X_loan_train, y_loan_train)

# 4. Email Spam dataset
print("\n=== Processing Email Spam Dataset ===")
X_email = processed_email.drop(columns=['Email No.', 'Prediction'])
y_email = processed_email['Prediction']
X_email_sel, email_features = select_features(X_email, y_email, k=10)
X_email_tr, X_email_val, X_email_te, y_email_tr, y_email_val, y_email_te = split_dataset(X_email_sel, y_email)

# 5. MNIST dataset
print("\n=== Processing MNIST Dataset ===")
X_mnist_train_full = processed_mnist['train_images']
y_mnist_train_full = processed_mnist['train_labels']
X_mnist_te = processed_mnist['test_images']
y_mnist_te = processed_mnist['test_labels']
X_mnist_tr, X_mnist_val, y_mnist_tr, y_mnist_val = train_test_split(
    X_mnist_train_full, y_mnist_train_full, test_size=0.15, random_state=42, stratify=y_mnist_train_full
)
print(f"MNIST splits - Train: {X_mnist_tr.shape[0]}, Val: {X_mnist_val.shape[0]}, Test: {X_mnist_te.shape[0]}")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

def evaluate_model(model, X_train, y_train, X_val, y_val, X_test, y_test, dataset_name):
    """
    Trains a model and evaluates it on Train, Val, and Test splits.
    """
    model.fit(X_train, y_train)
    
    # Predict
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)
    
    # Calculate Metrics
    val_acc = accuracy_score(y_val, y_val_pred)
    val_f1 = f1_score(y_val, y_val_pred, average='weighted')
    
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, average='weighted')
    
    print(f"=== Performance: {dataset_name} ===")
    print(f"Validation Accuracy: {val_acc:.4f} | Validation F1-Score: {val_f1:.4f}")
    print(f"Test Accuracy:       {test_acc:.4f} | Test F1-Score:       {test_f1:.4f}")
    print("\nTest Classification Report:")
    print(classification_report(y_test, y_test_pred))
    
    # Plot Confusion Matrix
    cm = confusion_matrix(y_test, y_test_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f"Confusion Matrix - {dataset_name} (Test)")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()
    
    return {
        'Dataset': dataset_name,
        'Val Accuracy': val_acc,
        'Val F1': val_f1,
        'Test Accuracy': test_acc,
        'Test F1': test_f1
    }

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier

results = []

# 1. Iris: KNN
print("Training KNN on Iris...")
iris_model = KNeighborsClassifier(n_neighbors=5)
iris_res = evaluate_model(iris_model, X_iris_tr, y_iris_tr, X_iris_val, y_iris_val, X_iris_te, y_iris_te, "Iris (KNN)")
results.append(iris_res)

# 2. Diabetes: Decision Tree
print("\nTraining Decision Tree on Diabetes...")
diab_model = DecisionTreeClassifier(max_depth=5, random_state=42)
diab_res = evaluate_model(diab_model, X_diab_tr, y_diab_tr, X_diab_val, y_diab_val, X_diab_te, y_diab_te, "Diabetes (Decision Tree)")
results.append(diab_res)

# 3. Loan: Random Forest
print("\nTraining Random Forest on Loan...")
loan_model = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42)
loan_res = evaluate_model(loan_model, X_loan_tr, y_loan_tr, X_loan_val, y_loan_val, X_loan_te, y_loan_te, "Loan (Random Forest)")
results.append(loan_res)

# 4. Email Spam: Decision Tree
print("\nTraining Decision Tree on Email Spam...")
email_model = DecisionTreeClassifier(max_depth=7, random_state=42)
email_res = evaluate_model(email_model, X_email_tr, y_email_tr, X_email_val, y_email_val, X_email_te, y_email_te, "Email Spam (Decision Tree)")
results.append(email_res)

# 5. MNIST: SGD Classifier (Linear SVM/Logistic Regression)
print("\nTraining SGD Classifier on MNIST...")
mnist_model = SGDClassifier(max_iter=20, random_state=42)
mnist_res = evaluate_model(mnist_model, X_mnist_tr, y_mnist_tr, X_mnist_val, y_mnist_val, X_mnist_te, y_mnist_te, "MNIST (SGD Classifier)")
results.append(mnist_res)

# Summary Performance Table
print("\n=== SUMMARY OF PERFORMANCE METRICS ===")
summary_df = pd.DataFrame(results)
print(summary_df.to_string(index=False))